# 16 · Can bibliographic coupling serve as a label?

**Post-hoc, exploratory** (added 2026-10-19, after notebooks 14–15). It supports a discussion point in `audit/PAPER_NOTES.md` and is not part of the pre-registration in notebook 14. It reuses notebook 14's pre-registered predictions (`v2/goldstandard/predictions/`, sample `d_20_1..10`) and computes no new scores.

**Questions, with the analysis fixed before running:**
- **(a) Precision of strong coupling.** When coupling is strong, do the human ratings agree?
  - *a1, pooled:* rated items (participant, paper, sample) in three coupling bins: 0, > 0, and the top decile of the non-zero scores. For each bin: mean rating and share rated ≥ 4. For reference, the same number of top items by Qwen3-Embedding. Pooling mixes participants, whose score scales differ, so this is descriptive.
  - *a2, within participant:* the mean human rating of each participant's top-1 rated paper by coupling vs by Qwen3 (tied tops are averaged), averaged over samples. Baseline: the participant's mean rating (a random pick). Paired Wilcoxon over participants.
- **(b) Coupling as the label.** Score the embeddings, and the human ratings themselves, against coupling scores used as labels (same weighted Kendall-tau loss; labels per profile sample; pairs with equal coupling carry no weight). Does the method order change compared with the human labels?

In [1]:
GOLDSTANDARD_DIR = ""   # absolute path to the gold-standard clone; papermill -p, or env GOLDSTANDARD_DIR
N_BOOT = 1000
SEED = 0

In [2]:
# Parameters
GOLDSTANDARD_DIR = "/Users/senaoz/Developer/SWE-599/research/data/external/goldstandard"


In [3]:
import json
import os
import sys
from pathlib import Path

RESEARCH = Path.cwd().parent if Path.cwd().name == "v2" else Path.cwd()
sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
pd.set_option("display.width", 200)

from src import bibcoupling as bc

G = Path(GOLDSTANDARD_DIR or os.environ["GOLDSTANDARD_DIR"])
PRED = RESEARCH / "v2" / "goldstandard" / "predictions"
OUT = RESEARCH / "v2" / "goldstandard"
SAMPLES = range(1, 11)

ev = pd.read_csv(G / "data" / "evaluations.csv", sep="\t")
refs = {str(r.ParticipantID): {r[f"Paper{i}"]: float(r[f"Expertise{i}"]) for i in range(1, 11)
                                if isinstance(r[f"Paper{i}"], str) and not pd.isna(r[f"Expertise{i}"])}
        for _, r in ev.iterrows()}
REVIEWERS = sorted(refs)
load = lambda algo: [json.load(open(PRED / f"{algo}_d_20_{i}_ta.json")) for i in SAMPLES]
P = {m: load(a) for m, a in [("coupling", "coupling-idf-cosine-max"), ("qwen", "qwen"),
                             ("specter2", "specter2"), ("minilm", "minilm")]}
items = pd.DataFrame([{"participant": rv, "paper": p, "sample": i, "rating": refs[rv][p],
                       "coupling": P["coupling"][i - 1][rv][p], "qwen": P["qwen"][i - 1][rv][p]}
                      for i in SAMPLES for rv in REVIEWERS for p in refs[rv]])
print(f"items {len(items)}, with coupling > 0: {(items.coupling > 0).mean():.1%}")

items 4770, with coupling > 0: 70.2%


## (a1) Pooled precision of strong coupling

In [4]:
rng = np.random.default_rng(SEED)
pools = [rng.choice(REVIEWERS, len(REVIEWERS), replace=True) for _ in range(N_BOOT)]
items["high"] = (items.rating >= 4).astype(float)
by_p = {rv: g for rv, g in items.groupby("participant")}

def stat_ci(mask_fn, col):
    """Point estimate and participant-bootstrap CI of mean(col) over the items selected by mask_fn."""
    sel = items[mask_fn(items)]
    point = sel[col].mean()
    boots = []
    for pool in pools:
        d = pd.concat([by_p[rv] for rv in pool])
        boots.append(d[mask_fn(d)][col].mean())
    return point, *np.nanpercentile(boots, [2.5, 97.5])

nz = items.coupling[items.coupling > 0]
c_cut = nz.quantile(0.9)
n_top = int((items.coupling >= c_cut).sum())
q_cut = items.qwen.sort_values(ascending=False).iloc[n_top - 1]
bins = {
    "all items (baseline)": lambda d: d.coupling > -1,
    "coupling = 0": lambda d: d.coupling == 0,
    "coupling > 0": lambda d: d.coupling > 0,
    f"coupling top decile of non-zero (≥ {c_cut:.3f})": lambda d: d.coupling >= c_cut,
    f"Qwen3, same number of top items (≥ {q_cut:.3f})": lambda d: d.qwen >= q_cut,
}
a1 = []
for name, fn in bins.items():
    r, rlo, rhi = stat_ci(fn, "rating")
    h, hlo, hhi = stat_ci(fn, "high")
    a1.append({"bin": name, "items": int(fn(items).sum()), "mean rating": r, "lo": rlo, "hi": rhi,
               "share ≥4": h, "share lo": hlo, "share hi": hhi})
a1 = pd.DataFrame(a1)
a1.round(3)

,bin,items,mean rating,lo,hi,share ≥4,share lo,share hi
0,all items (baseline),4770,3.706,3.600,3.812,0.564,0.515,0.614
1,coupling = 0,1423,3.429,3.175,3.681,0.439,0.325,0.548
2,coupling > 0,3347,3.824,3.704,3.949,0.617,0.559,0.678
3,coupling top decile of non-zero (≥ 0.182),336,4.501,4.256,4.688,0.881,0.737,0.973
4,"Qwen3, same number of top items (≥ 0.773)",340,4.301,3.991,4.577,0.824,0.692,0.939


## (a2) Within-participant precision@1

In [5]:
def top1_rating(preds_i, rv):
    s = preds_i[rv]
    best = max(s.values())
    return np.mean([refs[rv][p] for p, v in s.items() if v == best])

a2 = pd.DataFrame({rv: {"coupling": np.mean([top1_rating(P["coupling"][i], rv) for i in range(10)]),
                        "qwen": np.mean([top1_rating(P["qwen"][i], rv) for i in range(10)]),
                        "random pick": np.mean(list(refs[rv].values()))} for rv in REVIEWERS}).T
summary_a2 = a2.mean().rename("mean rating of top-1")
tests_a2 = []
for a, b in [("coupling", "random pick"), ("qwen", "random pick"), ("coupling", "qwen")]:
    stat, p = wilcoxon(a2[a], a2[b])
    d = a2[a] - a2[b]
    tests_a2.append({"a": a, "b": b, "mean diff": d.mean(), "a better": int((d > 0).sum()),
                     "b better": int((d < 0).sum()), "ties": int((d == 0).sum()), "W": stat, "p": f"{p:.2e}"})
print(summary_a2.round(3).to_string())
pd.DataFrame(tests_a2).round(3)

coupling       4.386
qwen           4.371
random pick    3.754


,a,b,mean diff,a better,b better,ties,W,p
0,coupling,random pick,0.632,48,10,0,234.5,1.52e-06
1,qwen,random pick,0.617,46,12,0,215.5,7.22e-07
2,coupling,qwen,0.015,17,14,27,221.0,5.94e-01


## (b) Coupling scores as the label

In [6]:
coupling_labels = P["coupling"]   # one label dict per profile sample
human_as_pred = [refs] * 10
rows = []
for name, preds in [("qwen", P["qwen"]), ("specter2", P["specter2"]), ("minilm", P["minilm"]),
                    ("human ratings", human_as_pred)]:
    under_c = bc.bootstrap_ci(preds, coupling_labels, N_BOOT, SEED)
    under_h = bc.bootstrap_ci(preds, refs, N_BOOT, SEED) if name != "human ratings" else (0.0, 0.0, 0.0)
    rows.append({"method": name, "loss vs human labels": under_h[0], "loss vs coupling labels": under_c[0],
                 "coupling lo": under_c[1], "coupling hi": under_c[2]})
under_h_c = bc.bootstrap_ci(P["coupling"], refs, N_BOOT, SEED)
rows.append({"method": "coupling", "loss vs human labels": under_h_c[0], "loss vs coupling labels": 0.0,
             "coupling lo": 0.0, "coupling hi": 0.0})
b = pd.DataFrame(rows).set_index("method")
weighted = sum(1 for i in range(10) for rv in REVIEWERS if len(set(coupling_labels[i][rv].values())) > 1)
print(f"(participant, sample) cells where coupling labels order anything: {weighted} of {10 * len(REVIEWERS)}")
b.round(3)

(participant, sample) cells where coupling labels order anything: 566 of 580


,loss vs human labels,loss vs coupling labels,coupling lo,coupling hi
method,,,,
qwen,0.222,0.212,0.167,0.264
specter2,0.231,0.241,0.192,0.302
minilm,0.242,0.228,0.173,0.292
human ratings,0.000,0.245,0.188,0.307
coupling,0.299,0.000,0.000,0.000


In [7]:
order_h = b.drop(["human ratings", "coupling"]).sort_values("loss vs human labels").index.tolist()
order_c = b.drop(["human ratings", "coupling"]).sort_values("loss vs coupling labels").index.tolist()
print("embedding order under human labels:   ", order_h)
print("embedding order under coupling labels:", order_c)
a1.to_csv(OUT / "16_a1_pooled_precision.csv", index=False, float_format="%.6g")
a2.to_csv(OUT / "16_a2_top1_by_participant.csv", float_format="%.6g")
pd.DataFrame(tests_a2).to_csv(OUT / "16_a2_tests.csv", index=False)
b.reset_index().to_csv(OUT / "16_b_coupling_as_label.csv", index=False, float_format="%.6g")
print("written 16_*.csv")

embedding order under human labels:    ['qwen', 'specter2', 'minilm']
embedding order under coupling labels: ['qwen', 'minilm', 'specter2']
written 16_*.csv
